# S10 - SMA200 combined with the CAPE excess yield (ensemble)

**Hypothesis.** SMA200 (trend) and the S01 CAPE excess yield (valuation) use different information:
trend reacts to price, valuation to how expensive equities are versus bonds. If valuation adds
information *conditional on* the trend, a combination should beat SMA200 alone.

| Variant | Equity weight at each monthly check |
|---|---|
| and | 1 if close > SMA200 **and** excess yield > its expanding median, else 0 |
| or | 1 if close > SMA200 **or** excess yield > its expanding median, else 0 |
| avg | 0.5 x SMA200 signal + 0.5 x CAPE signal (0, 0.5 or 1) |

**Inputs and point-in-time lags** (identical to the benchmark and to S01)

| Signal | Source | Lag used |
|---|---|---|
| close vs 200-day SMA | daily S&P 500 price | at the decision close |
| ey = 1 / CAPE - GS10 / 100 vs its expanding median (120-month warm-up, median lagged 1 month) | Shiller monthly | month M usable from M+2 (`bt_pit.RELEASE_LAGS["SHILLER_CAPE"]`) |

Monthly check, next-close execution, 5 bp per unit of turnover. Shiller data end 2024-09, so the
CAPE leg is frozen at its last value afterwards (as in S01).

**Sources.** S01 notebook (`valuation/cape_excess_yield.ipynb`); Asness (2003), *Fight the Fed model*,
SSRN 381480; Faber (2007), *A quantitative approach to tactical asset allocation*.

> Decision-support research only; never places orders.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
sys.path.insert(0, str(_root / "notebooks" / "backtest"))
import bt_common as bc  # noqa: E402
import bt_long  # noqa: E402
import bt_periods as bp  # noqa: E402
import bt_pit  # noqa: E402
import bt_registry as br  # noqa: E402
import bt_stats as bs  # noqa: E402
import seed_registry  # noqa: E402
import stage2 as s2  # noqa: E402

bc.setup_plotly("cdn")
pd.set_option("display.width", 180)
TRIAL_ID, STRATEGY, NOTEBOOK = 'S10', 'sma200_cape', 'ensemble/sma200_cape.ipynb'
PROTOCOL, REBALANCE = 'main', 'monthly'

sample = bt_long.long_sample()
px, cash = sample["px"], sample["cash"]
bench = bp.benchmark_returns(px, cash)  # SMA200 monthly / B&H / 60-40 / T-bills, frozen settings
print("prices", px.index[0].date(), "->", px.index[-1].date(), "| registry trials so far:", br.trial_count())

prices 1934-02-01 -> 2026-10-19 | registry trials so far: 31


## 1. Data and point-in-time inputs

In [2]:
inputs = {"px": px, "shiller": sample["shiller"][["CAPE", "GS10"]].dropna()}
print("Shiller CAPE/GS10", inputs["shiller"].index[0], "->", inputs["shiller"].index[-1])

Shiller CAPE/GS10 1881-01 -> 2024-09


## 2. Signal and variant grid (fixed before any backtest)

In [3]:
WARMUP_MONTHS = 120
LAG = bt_pit.RELEASE_LAGS["SHILLER_CAPE"]


def cape_signal(inputs):
    m = inputs["shiller"].copy()
    m.index = m.index.to_timestamp(how="start")
    ey = 1.0 / m["CAPE"] - m["GS10"] / 100.0
    median = ey.expanding(min_periods=WARMUP_MONTHS).median().shift(1)
    sig = (ey > median).astype(float).where(median.notna())
    return bt_pit.monthly_to_daily_pit(sig, inputs["px"].index, **LAG)


def build_signal(inputs, combo="and"):
    sma_on, _ = bc.sma_signal(inputs["px"]["close"], 200)
    cape_on = cape_signal(inputs)
    if combo == "and":
        return sma_on * cape_on
    if combo == "or":
        return np.maximum(sma_on, cape_on).where(sma_on.notna() & cape_on.notna())
    return 0.5 * sma_on + 0.5 * cape_on


VARIANTS = {c: dict(combo=c) for c in ("and", "or", "avg")}
PARAMS_TEXT = {n: f"{n}; SMA200 x CAPE excess yield (S01 rule, pit M+2); monthly; next_close; 5bp" for n in VARIANTS}

## 3. Look-ahead checks

Every variant is recomputed with all inputs cut at past dates (monthly inputs: completed months only) and must equal the full-data signal on those dates (`stage2.truncation_test`). No backtest runs unless all pass.

In [4]:
N_CUTS = 30
trunc = {}
for name, params in VARIANTS.items():
    trunc[name] = s2.truncation_test(build_signal, inputs, n_cuts=N_CUTS, **params)
display(pd.DataFrame({k: v.row() for k, v in trunc.items()}).T)
bad = [k for k, v in trunc.items() if not v.passed]
assert not bad, f"look-ahead detected in variants {bad}: fix before any backtest"
print("all variants pass the truncation look-ahead test")

,cuts,mismatches,max_abs_diff,passed
and,30,0,0.0,True
or,30,0,0.0,True
avg,30,0,0.0,True


all variants pass the truncation look-ahead test


## 4. Development-period sweep

Each variant is scored on the **dev** period only and registered as a trial. The best dev Sharpe is chosen; validation and holdout results are never used for the choice.

In [5]:
signals = {name: build_signal(inputs, **params) for name, params in VARIANTS.items()}
sweep, CHOSEN = s2.dev_sweep(px, cash, bench, signals, PARAMS_TEXT, trial_id=TRIAL_ID, strategy=STRATEGY,
                             notebook=NOTEBOOK, protocol=PROTOCOL, rebalance=REBALANCE)
display(sweep)
print("chosen on dev Sharpe only:", CHOSEN, "|", PARAMS_TEXT[CHOSEN])

,dev_sharpe,d_sma200,d_bh,time_in_market,switches_pa,dev_start
and,0.499602,-0.144133,-0.011396,0.229676,0.720998,1935-11-26
or,0.676141,0.032406,0.165142,0.770029,1.109227,1935-11-26
avg,0.690684,0.046948,0.179685,0.499853,1.811738,1935-11-26


chosen on dev Sharpe only: avg | avg; SMA200 x CAPE excess yield (S01 rule, pit M+2); monthly; next_close; 5bp


## 5. Frozen evaluation of the chosen variant

Frozen primary metric (`PLAN.md` section 2): beat SMA200 and buy & hold in dev **and** val, one-sided block-bootstrap p <= 0.10 vs SMA200 in both, deflated Sharpe >= 0.95 with the registry trial count. Holdout is confirmatory only.

In [6]:
signal = signals[CHOSEN]
runs, cmp, stats, res = s2.evaluate(px, cash, bench, signal, protocol=PROTOCOL, rebalance=REBALANCE)
display(cmp.round(3))
display(stats)
display(res.to_frame().round(3))
print(f"n_trials={res.n_trials} | DSR={res.dsr:.3f} | passes={res.passes} | holdout confirms={res.holdout_confirms}")
display(pd.Series(res.checks, name="check"))

,strategy,bh,d_bh,sma200,d_sma200,mix60,d_mix60
dev,0.691,0.511,0.180,0.644,0.047,0.518,0.173
val,0.555,0.319,0.236,0.572,-0.017,0.309,0.246
holdout,0.760,0.781,-0.021,0.658,0.103,0.779,-0.019
full,0.675,0.513,0.162,0.630,0.045,0.515,0.160


,CAGR,Vol,Sharpe,Sortino,Max DD,Calmar,Worst day,MaxDD peak,MaxDD recovered,Final x,Time in market,Switches,Switches / yr,Cost drag / yr
dev,0.101318,0.089972,0.690684,0.956922,-0.30207,0.335412,-0.101998,1938-10-10,1943-03-04,184.982336,0.499853,98,1.811738,0.000458
val,0.074892,0.067935,0.554888,0.792432,-0.151325,0.494905,-0.034125,2007-10-09,2009-06-01,4.237611,0.373785,24,1.200329,0.0003
holdout,0.075733,0.081387,0.760107,1.062108,-0.127426,0.594332,-0.058829,2022-01-03,2024-03-01,3.40623,0.502604,35,2.08476,0.000536
full,0.090679,0.083994,0.674978,0.940531,-0.30207,0.300192,-0.101998,1938-10-10,1943-03-04,2670.086391,0.472535,157,1.727236,0.000437


,Sharpe,d_sma200,d_bh,p_vs_sma200
dev,0.691,0.047,0.180,0.247
val,0.555,-0.017,0.236,0.646
holdout,0.760,0.103,-0.021,NaN
full,0.675,0.045,0.162,NaN


n_trials=34 | DSR=1.000 | passes=False | holdout confirms=True


dev_beats_sma200         True
dev_beats_bh             True
dev_p_le_0.1            False
val_beats_sma200        False
val_beats_bh             True
val_p_le_0.1            False
dsr_ge_0.95              True
holdout_d_sma200_pos     True
Name: check, dtype: bool

In [7]:
s2.plot_growth(runs, bench, f'{TRIAL_ID} {CHOSEN}', PROTOCOL).show()

## 6. Robustness: costs, placebo, peek test, decades

In [8]:
cs = bs.cost_sensitivity(px, signal, cash, rebalance=REBALANCE)
display(cs.round(3))
sma_full = bs.excess_sharpe(bench["sma200"].reindex(runs["full"].index), bench["tbill"].reindex(runs["full"].index))
be = bs.breakeven_cost_bps(px, signal, cash, bench_sharpe=sma_full, rebalance=REBALANCE)
print(f"break-even one-way cost vs SMA200 Sharpe over the same window ({sma_full:.3f}): {be:.1f} bp")
pl = bs.placebo_test(px, signal, cash, n_placebo=200, rebalance=REBALANCE, rng=np.random.default_rng(bs.SEED))
display(pd.Series(pl, name="placebo").to_frame())
peek = bt_pit.lookahead_test(px, signal, cash, rebalance=REBALANCE)
print(f"peek test: Sharpe by shift {peek.sharpe_by_shift.round(3).to_dict()} | gain {peek.peek_gain:+.3f}")
display(s2.decade_table(runs["full"], bench).style.format("{:.2f}"))

,Sharpe,CAGR,Max DD,turnover_pa
cost_bps,,,,
0,0.704,0.095,-0.301,0.878
5,0.699,0.094,-0.302,0.878
10,0.694,0.094,-0.303,0.878
25,0.678,0.092,-0.307,0.878
50,0.653,0.090,-0.313,0.878


break-even one-way cost vs SMA200 Sharpe over the same window (0.630): 72.4 bp


,placebo
stat,Sharpe
real,0.698743
placebo_mean,0.2101
placebo_q95,0.379222
quantile,1.0
p_one_sided,0.0
n_placebo,200
time_in_market,0.171803


peek test: Sharpe by shift {-21: 1.127, -5: 0.803, -1: 0.686, 0: 0.699, 1: 0.697, 5: 0.675, 21: 0.679} | gain +0.428


,Sharpe,Sharpe B&H,Sharpe SMA200,CAGR,CAGR B&H,CAGR SMA200,time in market
1930s,0.48,0.26,0.41,0.07,0.04,0.06,0.61
1940s,0.64,0.61,0.57,0.08,0.09,0.06,0.78
1950s,1.65,1.42,1.51,0.17,0.19,0.18,0.68
1960s,0.43,0.40,0.43,0.06,0.08,0.07,0.35
1970s,0.53,0.03,0.37,0.09,0.06,0.09,0.28
1980s,0.58,0.53,0.58,0.13,0.17,0.17,0.36
1990s,0.64,0.92,0.64,0.09,0.18,0.13,0.42
2000s,0.49,-0.05,0.50,0.06,-0.01,0.08,0.33
2010s,0.81,0.90,0.71,0.07,0.14,0.09,0.53
2020s,0.69,0.67,0.58,0.09,0.15,0.10,0.46


## 7. Register the result

In [9]:
note = ("passes" if res.passes else "fails") + f" frozen metric; chosen {CHOSEN} on dev; protocol {PROTOCOL}"
rows = s2.register_final(TRIAL_ID, STRATEGY, NOTEBOOK, PARAMS_TEXT[CHOSEN], res, note=note, header=seed_registry.HEADER)
display(rows)
print("registry trial count:", br.trial_count())

,trial_id,date,strategy,notebook,params,period,sharpe,d_sma200,d_bh,p_vs_sma200,dsr,n_trials,passes,status,note
78,S10-v1/dev,2026-10-20,sma200_cape,ensemble/sma200_cape.ipynb,"and; SMA200 x CAPE excess yield (S01 rule, pit...",dev,0.4996,-0.1441,-0.0114,,,,,run,"dev-only sweep, variant and"
79,S10-v2/dev,2026-10-20,sma200_cape,ensemble/sma200_cape.ipynb,"or; SMA200 x CAPE excess yield (S01 rule, pit ...",dev,0.6761,0.0324,0.1651,,,,,run,"dev-only sweep, variant or"
80,S10-v3/dev,2026-10-20,sma200_cape,ensemble/sma200_cape.ipynb,"avg; SMA200 x CAPE excess yield (S01 rule, pit...",dev,0.6907,0.0469,0.1797,,,,,run,"dev-only sweep, variant avg"
81,S10/dev,2026-10-20,sma200_cape,ensemble/sma200_cape.ipynb,"avg; SMA200 x CAPE excess yield (S01 rule, pit...",dev,0.6907,0.0469,0.1797,0.2473,,34,False,fail,fails frozen metric; chosen avg on dev; protoc...
82,S10/val,2026-10-20,sma200_cape,ensemble/sma200_cape.ipynb,"avg; SMA200 x CAPE excess yield (S01 rule, pit...",val,0.5549,-0.0173,0.2358,0.6465,,34,False,fail,fails frozen metric; chosen avg on dev; protoc...
83,S10/holdout,2026-10-20,sma200_cape,ensemble/sma200_cape.ipynb,"avg; SMA200 x CAPE excess yield (S01 rule, pit...",holdout,0.7601,0.1026,-0.0206,,,34,False,fail,fails frozen metric; chosen avg on dev; protoc...
84,S10/full,2026-10-20,sma200_cape,ensemble/sma200_cape.ipynb,"avg; SMA200 x CAPE excess yield (S01 rule, pit...",full,0.6750,0.0453,0.1617,,1.0000,34,False,fail,fails frozen metric; chosen avg on dev; protoc...


registry trial count: 34


## 8. Current signal

In [10]:
last = signal.dropna()
print(f"latest signal ({CHOSEN}) on {last.index[-1].date()}: equity weight {last.iloc[-1]:.2f}")
chk = bc.check_days(px.index, REBALANCE)
display(last[chk.reindex(last.index).fillna(False)].tail(6).to_frame("equity weight at decision"))
print("Research output only -- not a recommendation or an order.")

latest signal (avg) on 2026-10-19: equity weight 0.50


,equity weight at decision
Date,
2026-04-30,0.5
2026-05-29,0.5
2026-06-30,0.5
2026-07-31,0.5
2026-08-31,0.5
2026-09-30,0.5


Research output only -- not a recommendation or an order.


## 9. Evidence summary (computed)

In [11]:
for line in s2.evidence_lines(res, trunc[CHOSEN], peek, pl, be):
    print("*", line)

* Truncation look-ahead test: PASSED on 30 cut dates; peek test gain +0.428.
* dev: Sharpe 0.691 (vs SMA200 +0.047, vs B&H +0.180, p vs SMA200 0.247)
* val: Sharpe 0.555 (vs SMA200 -0.017, vs B&H +0.236, p vs SMA200 0.646)
* holdout: Sharpe 0.760 (vs SMA200 +0.103, vs B&H -0.021)
* Deflated Sharpe 1.000 with n_trials=34.
* Placebo: real 0.699 vs placebo 95th pct 0.379 (p 0.000); break-even cost vs SMA200 full 72.4 bp.
* Verdict: FAILS the frozen primary metric (failed checks: dev_p_le_0.1, val_beats_sma200, val_p_le_0.1); holdout confirms.


## 10. Caveats

* The CAPE leg is the S01 rule unchanged; its long rate before 1953 is Shiller's series, which is
  interpolated from annual data (a small potential look-ahead that S02 avoided by using Moody's AAA).
* S01 alone sat in equities only ~26% of the time, so `and` is a very defensive combination.
* SMA200 itself is the benchmark: a combination can only pass if valuation adds information that the
  trend does not already contain.

## 11. Interpretation

* **The closest any idea came to passing.** The 50/50 blend (`avg`) beats SMA200 in dev (+0.05 Sharpe) and in
  the holdout (+0.10), and is level in val (-0.02). It is far above random timing (placebo p 0.000) and its
  edge survives costs (break-even ~72 bp). But none of the differences vs SMA200 is significant
  (bootstrap p 0.25 dev, 0.65 val), so it **fails** the frozen metric.
* The higher Sharpe comes from **lower risk, not higher return**: full sample CAGR 9.1% vs SMA200 10.5% and
  B&H 11.0%, volatility 8.4% vs 11.5%, max drawdown -30% vs -33%, about 47% average equity weight. In the
  holdout bull market it earned 7.6% a year vs 9.4% (SMA200) and 14.3% (B&H). An investor targeting
  return rather than Sharpe would not prefer it.
* `and` (both signals on) is far too defensive (23% invested) and lost to B&H in dev; `or` was close to `avg`.
* Valuation adds a little information conditional on trend, mostly by cutting exposure in expensive
  markets (1970s decade Sharpe 0.53 vs SMA200 0.37). The CAPE leg inherits S01's pre-1953 Shiller long
  rate caveat and is frozen after 2024-09.